In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib
import json

In [ ]:
df = pd.read_csv("salary_prediction_data.csv")

print("Dataset shape:", df.shape)
df.head()

In [ ]:
print(df.info())


In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

In [ ]:
print(df.describe())

In [ ]:
df.columns = df.columns.str.strip()


In [ ]:
text_columns = [
    "Gender",
    "Education Level",
    "Job Title"
]

for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

In [ ]:
numeric_columns = [
    "Age",
    "Years of Experience",
    "Salary"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:
df = df.dropna(
    subset=[
        "Age",
        "Gender",
        "Education Level",
        "Job Title",
        "Years of Experience",
        "Salary"
    ]
)

In [ ]:
df = df.drop_duplicates()

print("Cleaned dataset shape:", df.shape)

In [ ]:
job_titles = sorted(df["Job Title"].unique())

print("Number of job titles:", len(job_titles))

for title in job_titles:
    print(title)

In [ ]:
print(df["Job Title"].value_counts())

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(df["Salary"], bins=30)

plt.xlabel("Salary")
plt.ylabel("Number of Employees")
plt.title("Salary Distribution")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.scatter(
    df["Years of Experience"],
    df["Salary"],
    alpha=0.5
)

plt.xlabel("Years of Experience")
plt.ylabel("Salary")
plt.title("Years of Experience vs Salary")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

df.groupby("Education Level")["Salary"].mean().sort_values().plot(
    kind="bar"
)

plt.xlabel("Education Level")
plt.ylabel("Average Salary")
plt.title("Average Salary by Education Level")

plt.xticks(rotation=45)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=df,
    x="Gender",
    y="Salary"
)

plt.title("Salary Distribution by Gender")
plt.show()

In [ ]:
X = df[
    [
        "Age",
        "Gender",
        "Education Level",
        "Job Title",
        "Years of Experience"
    ]
]

y = df["Salary"]

In [ ]:
print("Features:")
print(X.columns.tolist())

print("\nTarget:")
print(y.name)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

In [ ]:
numeric_features = [
    "Age",
    "Years of Experience"
]

categorical_features = [
    "Gender",
    "Education Level",
    "Job Title"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

In [ ]:
linear_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear_pipeline.fit(X_train, y_train)

linear_pred = linear_pipeline.predict(X_test)

In [ ]:
linear_mae = mean_absolute_error(y_test, linear_pred)

linear_rmse = np.sqrt(
    mean_squared_error(y_test, linear_pred)
)

linear_r2 = r2_score(y_test, linear_pred)

print("Linear Regression")
print("--------------------")
print("MAE :", linear_mae)
print("RMSE:", linear_rmse)
print("R²  :", linear_r2)

In [ ]:
tree_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        DecisionTreeRegressor(
            random_state=42,
            max_depth=15
        )
    )
])

tree_pipeline.fit(X_train, y_train)

tree_pred = tree_pipeline.predict(X_test)

In [ ]:
tree_mae = mean_absolute_error(y_test, tree_pred)

tree_rmse = np.sqrt(
    mean_squared_error(y_test, tree_pred)
)

tree_r2 = r2_score(y_test, tree_pred)

print("Decision Tree")
print("--------------------")
print("MAE :", tree_mae)
print("RMSE:", tree_rmse)
print("R²  :", tree_r2)

In [ ]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        RandomForestRegressor(
            n_estimators=200,
            random_state=42,
            max_depth=20,
            n_jobs=-1
        )
    )
])

rf_pipeline.fit(X_train, y_train)

rf_pred = rf_pipeline.predict(X_test)

In [ ]:
rf_mae = mean_absolute_error(y_test, rf_pred)

rf_rmse = np.sqrt(
    mean_squared_error(y_test, rf_pred)
)

rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest")
print("--------------------")
print("MAE :", rf_mae)
print("RMSE:", rf_rmse)
print("R²  :", rf_r2)

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "MAE": [
        linear_mae,
        tree_mae,
        rf_mae
    ],
    "RMSE": [
        linear_rmse,
        tree_rmse,
        rf_rmse
    ],
    "R2": [
        linear_r2,
        tree_r2,
        rf_r2
    ]
})

results

In [ ]:
error_df = X_test.copy()

error_df["Actual Salary"] = y_test.values
error_df["Predicted Salary"] = rf_pred

error_df["Error"] = (
    error_df["Actual Salary"]
    - error_df["Predicted Salary"]
)

error_df["Absolute Error"] = abs(error_df["Error"])

error_df.head(10)

In [ ]:
error_df.sort_values(
    "Absolute Error",
    ascending=False
).head(20)

In [ ]:
job_error = error_df.groupby("Job Title").agg(
    Average_Actual_Salary=("Actual Salary", "mean"),
    Average_Predicted_Salary=("Predicted Salary", "mean"),
    Average_Absolute_Error=("Absolute Error", "mean"),
    Number_of_Records=("Actual Salary", "count")
).sort_values(
    "Average_Absolute_Error",
    ascending=False
)

job_error

In [ ]:
print(
    "Unique RF predictions:",
    len(np.unique(np.round(rf_pred, 2)))
)

print(
    "Total predictions:",
    len(rf_pred)
)

In [ ]:
prediction_counts = pd.Series(
    np.round(rf_pred, 2)
).value_counts()

prediction_counts.head(20)

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    y_test,
    rf_pred,
    alpha=0.5
)

plt.xlabel("Actual Salary")
plt.ylabel("Predicted Salary")
plt.title("Actual vs Predicted Salary")

plt.show()

In [ ]:
final_pipeline = rf_pipeline

print("Final model: Random Forest")

In [ ]:
new_employee = pd.DataFrame({
    "Age": [25],
    "Gender": ["Male"],
    "Education Level": ["Bachelor's"],
    "Job Title": ["Software Engineer"],
    "Years of Experience": [2]
})

prediction = final_pipeline.predict(new_employee)

print("Predicted Salary:", prediction[0])

In [ ]:
import joblib

joblib.dump(final_pipeline, "salary_model.pkl")

print("salary_model.pkl saved successfully!")

In [ ]:
loaded_model = joblib.load("salary_model.pkl")

test_prediction = loaded_model.predict(new_employee)

print("Prediction from saved model:", test_prediction[0])

In [ ]:
valid_job_titles = sorted(df["Job Title"].unique().tolist())

print("Number of valid job titles:", len(valid_job_titles))
print(valid_job_titles)

In [ ]:
import json

with open("valid_job_titles.json", "w") as f:
    json.dump(valid_job_titles, f, indent=4)

print("valid_job_titles.json saved successfully!")

In [ ]:
from google.colab import files

files.download("salary_model.pkl")
files.download("valid_job_titles.json")